In [ ]:
!pip install scikit-learn
!pip install tensorflow
!pip install torch
!pip install xgboost

In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import vstack
import xgboost as xgb

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_json('C:/Users/007pe/Downloads/tdidf_embeddings.json', lines=True)

In [ ]:
df = df.dropna(subset=['Speaker_party_name'])
print("DataFrame after dropping NaN in 'Speaker_party_name':")
print(df)

In [ ]:
X = np.vstack(df['embedding'].values)
y = df['Speaker_party_name']

In [ ]:
encoder = LabelEncoder()

y = encoder.fit_transform(y)  
num_classes = len(encoder.classes_) 
print(encoder.classes_)
print(num_classes)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

In [ ]:
xgb_classifier = xgb.XGBClassifier(objective='multi:softmax', num_class=num_classes, random_state=0)

In [ ]:
xgb_classifier.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=True)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(xgb_classifier, X_train, y_train, cv=cv, scoring='accuracy')

print("Cross-validation results:")
print(f"Mean Accuracy: {np.mean(scores):.4f} \u00b1 {np.std(scores):.4f}")

# Make final predictions
print("\nMaking predictions...")
predictions = xgb_classifier.predict(X_test)
proba = xgb_classifier.predict_proba(X_test)

# Evaluate
print("\nClassification Report:")
print(classification_report(y_test, predictions, zero_division=0))
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))